In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip  # se tiverem delta instalado via pip

builder = (
    SparkSession.builder
    .appName("Spark_DFs_SQL_Tourism")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")  # se for preciso trazer o jar
    .enableHiveSupport()
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession com Delta inicializada!")
print("Versão do Spark:", spark.version)

✔️ SparkSession com Delta inicializada!
Versão do Spark: 3.4.1


In [2]:
from pyspark.sql.functions import col

DELTA_PATH = "hdfs://hdfs-nn:9000/demo/silver/tourism_silver/"

df_silver = (
    spark.read
        .format("delta")
        .load(DELTA_PATH)
)

df_silver.printSchema()
df_silver.show(20, truncate=False)

root
 |-- country_name: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- arrivals: long (nullable = true)

+----------------------+------------+----+--------+
|country_name          |country_code|year|arrivals|
+----------------------+------------+----+--------+
|Aruba                 |ABW         |2006|694000  |
|Angola                |AGO         |2006|121000  |
|Albania               |ALB         |2006|937000  |
|Argentina             |ARG         |2006|4173000 |
|Armenia               |ARM         |2006|382000  |
|Antigua and Barbuda   |ATG         |2006|254000  |
|Australia             |AUS         |2006|5532000 |
|Austria               |AUT         |2006|20269000|
|Belgium               |BEL         |2006|6995000 |
|Benin                 |BEN         |2006|180000  |
|Bulgaria              |BGR         |2006|5158000 |
|Bahamas, The          |BHS         |2006|1601000 |
|Bosnia and Herzegovina|BIH         |2006|256000  |

In [3]:
df_silver.createOrReplaceTempView("tourism_silver")

spark.sql("""
    SELECT 
        country_name,
        country_code,
        year,
        arrivals
    FROM tourism_silver
    ORDER BY country_code, year
    LIMIT 20
""").show(truncate=False)

+------------+------------+----+--------+
|country_name|country_code|year|arrivals|
+------------+------------+----+--------+
|Aruba       |ABW         |2003|642000  |
|Aruba       |ABW         |2004|728000  |
|Aruba       |ABW         |2005|733000  |
|Aruba       |ABW         |2006|694000  |
|Aruba       |ABW         |2007|772000  |
|Aruba       |ABW         |2008|827000  |
|Aruba       |ABW         |2009|813000  |
|Aruba       |ABW         |2010|824000  |
|Aruba       |ABW         |2011|869000  |
|Aruba       |ABW         |2012|904000  |
|Angola      |AGO         |2003|107000  |
|Angola      |AGO         |2004|194000  |
|Angola      |AGO         |2005|210000  |
|Angola      |AGO         |2006|121000  |
|Angola      |AGO         |2007|195000  |
|Angola      |AGO         |2008|294000  |
|Angola      |AGO         |2009|366000  |
|Angola      |AGO         |2010|425000  |
|Angola      |AGO         |2011|481000  |
|Angola      |AGO         |2012|528000  |
+------------+------------+----+--